In [4]:
#%pip install lightfm
#%pip install pandas
#%pip install matplotlib
#%pip install --upgrade pip
#%pip install lightfm

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import timedelta
from lightfm import LightFM

In [6]:
#Загрузим наши таблицы

df_users = pd.read_csv('KION_DATASET-main/data_original/users.csv')
df_items = pd.read_csv('KION_DATASET-main/data_original/items.csv')
df_interactions = pd.read_csv('KION_DATASET-main/interactions.csv')

In [7]:
# сначала просто посмотрим размеры таблиц

print('users:', df_users.shape)
print('items:', df_items.shape)
print('interactions:', df_interactions.shape)

users: (840197, 5)
items: (15963, 14)
interactions: (1594787, 5)


In [8]:
# первые строки interactions
# это главная таблица для рекомендаций:
# кто, что и когда смотрел

df_interactions.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250.0,72.0
1,699317,1659,2021-05-29,8317.0,100.0
2,656683,7107,2021-05-09,10.0,0.0
3,864613,7638,2021-07-05,14483.0,100.0
4,964868,9506,2021-04-30,6725.0,100.0


In [9]:
# первые строки users
# тут можно понять, какие признаки есть у пользователя

df_users.head()

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


In [10]:
# первые строки items
# тут уже видим признаки объектов: тип, жанры, год и т.д.

df_items.head()

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [11]:
# теперь посмотрим типы столбцов
# это нужно, чтобы понять, что у нас числовое, а что категориальное

print('users')
df_users.info()

print()
print('items')
df_items.info()

print()
print('interactions')
df_interactions.info()

users
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 840197 entries, 0 to 840196
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   user_id   840197 non-null  int64 
 1   age       826102 non-null  object
 2   income    825421 non-null  object
 3   sex       826366 non-null  object
 4   kids_flg  840197 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 32.1+ MB

items
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15963 entries, 0 to 15962
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   item_id       15963 non-null  int64  
 1   content_type  15963 non-null  object 
 2   title         15963 non-null  object 
 3   title_orig    11218 non-null  object 
 4   release_year  15865 non-null  float64
 5   genres        15963 non-null  object 
 6   countries     15926 non-null  object 
 7   for_kids      566 non-null    float64
 8   age_ratin

In [12]:
# теперь посмотрим пропуски
# это тоже полезно, чтобы понимать, где данные полные, а где нет

print('Пропуски в users')
print(df_users.isna().sum())
print()

print('Пропуски в items')
print(df_items.isna().sum())
print()

print('Пропуски в interactions')
print(df_interactions.isna().sum())

Пропуски в users
user_id         0
age         14095
income      14776
sex         13831
kids_flg        0
dtype: int64

Пропуски в items
item_id             0
content_type        0
title               0
title_orig       4745
release_year       98
genres              0
countries          37
for_kids        15397
age_rating          2
studios         14898
directors        1509
actors           2619
description         2
keywords          423
dtype: int64

Пропуски в interactions
user_id            0
item_id            0
last_watch_dt      0
total_dur          1
watched_pct      268
dtype: int64


In [13]:
# небольшая описательная статистика
# тут смотрим общую картину по числовым столбцам

print('users')
print(df_users.describe(include='all'))
print()

print('items')
print(df_items.describe(include='all'))
print()

print('interactions')
print(df_interactions.describe(include='all'))

users


             user_id        age        income     sex       kids_flg
count   8.401970e+05     826102        825421  826366  840197.000000
unique           NaN          6             6       2            NaN
top              NaN  age_25_34  income_20_40       Ж            NaN
freq             NaN     233926        471519  425270            NaN
mean    5.487668e+05        NaN           NaN     NaN       0.301106
std     3.168841e+05        NaN           NaN     NaN       0.458739
min     0.000000e+00        NaN           NaN     NaN       0.000000
25%     2.740990e+05        NaN           NaN     NaN       0.000000
50%     5.488080e+05        NaN           NaN     NaN       0.000000
75%     8.232380e+05        NaN           NaN     NaN       1.000000
max     1.097558e+06        NaN           NaN     NaN       1.000000

items
             item_id content_type  title    title_orig  release_year  \
count   15963.000000        15963  15963         11218  15865.000000   
unique           NaN 

Для LightFM нам в первую очередь нужна таблица interactions.

Основные столбцы для модели: user_id, item_id, last_watch_dt, watched_pct

users и items можно посмотреть для общего понимания датасета, но сами модели дальше будут строиться в основном по interactions.

In [14]:
# Посмотрим, сколько уникальных пользователей и item
# это важно для понимания масштаба задачи

print('Уникальных пользователей в interactions:', df_interactions['user_id'].nunique())
print('Уникальных item в interactions:', df_interactions['item_id'].nunique())
print('Всего взаимодействий:', len(df_interactions))

Уникальных пользователей в interactions: 567588
Уникальных item в interactions: 12693
Всего взаимодействий: 1594787


Предобработка данных

In [15]:
# теперь готовим данные
# переводим дату в datetime и убираем совсем слабые просмотры

df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')
df_interactions = df_interactions.dropna(subset=['last_watch_dt']).copy()

# оставим только более осмысленные просмотры
df_interactions = df_interactions[df_interactions['watched_pct'] > 50].copy()

print(df_interactions.shape)

(685055, 5)


In [16]:
# после фильтрации еще раз посмотрим,
# сколько осталось пользователей, item и взаимодействий

print('После фильтрации:')
print('Уникальных пользователей:', df_interactions['user_id'].nunique())
print('Уникальных item:', df_interactions['item_id'].nunique())
print('Всего взаимодействий:', len(df_interactions))

После фильтрации:
Уникальных пользователей: 308077
Уникальных item: 9722
Всего взаимодействий: 685055


In [17]:
# делим данные на train и test
# test = последние 10 дней

test_size_days = 10

max_date = df_interactions['last_watch_dt'].max()
border_date = max_date - timedelta(days=test_size_days)

df_train_all = df_interactions[df_interactions['last_watch_dt'] < border_date].copy()
df_test_all = df_interactions[df_interactions['last_watch_dt'] >= border_date].copy()

print('df_train_all:', df_train_all.shape)
print('df_test_all:', df_test_all.shape)
print('border_date:', border_date)

df_train_all: (617377, 5)
df_test_all: (67678, 5)
border_date: 2021-08-12 00:00:00


In [18]:
# для LightFM, как и для ALS, лучше брать полную историю train
# а не только последние 15 взаимодействий

# оставим только тех пользователей,
# у которых есть хотя бы 2 взаимодействия и в train, и в test

train_counts = df_train_all['user_id'].value_counts()
test_counts = df_test_all['user_id'].value_counts()

active_train_users = set(train_counts[train_counts >= 2].index)
active_test_users = set(test_counts[test_counts >= 2].index)

good_users = active_train_users & active_test_users

print('good_users:', len(good_users))

good_users: 5054


In [19]:
# теперь делаем уже финальные train и test
# только для good_users

df_train_lfm = df_train_all[df_train_all['user_id'].isin(good_users)].copy()
df_test = df_test_all[df_test_all['user_id'].isin(good_users)].copy()

print('df_train_lfm:', df_train_lfm.shape)
print('df_test:', df_test.shape)

df_train_lfm: (35107, 5)
df_test: (14743, 5)


In [20]:
# делаем test-таблицу:
# user_id и список item_id, которые реально были у пользователя в test

data_test_group = df_test.groupby('user_id')['item_id'].unique().reset_index()

print('test users:', data_test_group['user_id'].nunique())
data_test_group.head()

test users: 5054


,user_id,item_id
0,259,"[10509, 10772]"
1,655,"[11188, 5199]"
2,753,"[9327, 8350]"
3,835,"[5434, 11640, 10878]"
4,960,"[8636, 12770, 6809]"


In [21]:
# еще раз проверим, что все готово к моделям

print('train users:',df_train_lfm['user_id'].nunique())
print('train items:', df_train_lfm['item_id'].nunique())
print('test users:', data_test_group['user_id'].nunique())

train users: 5054
train items: 4642
test users: 5054


Реализация метрик

In [22]:

def hit_rate(recommended_list, bought_list):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(bought_list, recommended_list)
    
    hit_rate = (flags.sum() > 0) * 1
    
    return hit_rate


def hit_rate_at_k(recommended_list, bought_list, k=5):
    #my code
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list[:k])
    
    flags = np.isin(bought_list, recommended_list)
    
    hit_rate = (flags.sum() > 0) * 1
    
    return hit_rate

def precision(recommended_list, bought_list):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(bought_list, recommended_list)
    
    precision = flags.sum() / len(recommended_list)
    
    return precision


def precision_at_k(recommended_list, bought_list, k=5):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    #assert len(bought_list) > len(recommended_list)
    bought_list = bought_list  # Тут нет [:k] !!
    recommended_list = recommended_list[:k]
    
    flags = np.isin(bought_list, recommended_list)
    
    precision = flags.sum() / len(recommended_list)
    
    
    return precision

def money_precision_at_k(recommended_list, bought_list, prices_recommended, k=5):
        
    # my_code
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list[:k])
    flags = np.isin(recommended_list,bought_list)
    sm_bought=0
    for i in range(len(flags)):
        if flags[i]:
            sm_bought+=prices_recommended[i]
    m_precision=sm_bought/sum(prices_recommended)
    
    return m_precision

def recall(recommended_list, bought_list):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(bought_list, recommended_list)
    
    recall = flags.sum() / len(bought_list)
    
    return recall


def recall_at_k(recommended_list, bought_list, k=5):
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list[:k])
    
    flags = np.isin(bought_list, recommended_list)
    
    recall = flags.sum() / len(bought_list)
    
    return recall


def money_recall_at_k(recommended_list, bought_list, prices_recommended, prices_bought, k=5):
    prices_recommended=prices_recommended[:k]
    prince_recomend_relev=0
    for i in range(k):
        if recommended_list[i] in bought_list:
            prince_recomend_relev+=prices_recommended[i]
    m_recall=prince_recomend_relev/sum(prices_bought)
    
    
    return m_recall

def ap_k(recommended_list, bought_list, k=5):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(recommended_list, bought_list)
    
    if sum(flags) == 0:
        return 0
    
    sum_ = 0
    for i in range(0, k-1):
        if flags[i] == True:
            p_k = precision_at_k(recommended_list, bought_list, k=i+1)
            sum_ += p_k
            
    result = sum_ / sum(flags)
    
    return result


def map_k(recommended_list, bought_list, k=5, u=1):
    # my_code
    _sum=0
    for i in range(u):
        _sum+=ap_k(recommended_list[i], bought_list[i], k=k)
    result=_sum/u
    return result

In [23]:
# для оценки будем смотреть top-10 рекомендаций

k = 10

In [24]:
# Функция по объединенному рассчету метрик
# money-метрики не считаем, потому что у нас нет цен

def calc_metrics(result_df, rec_col, true_col='item_id', k=10):
    hit_rate_10 = 0
    precision_10 = 0
    recall_10 = 0

    for i in range(len(result_df)):
        rec_list = result_df[rec_col].iloc[i]
        true_list = result_df[true_col].iloc[i]

        hit_rate_10 += hit_rate_at_k(rec_list, true_list, k=k)
        precision_10 += precision_at_k(rec_list, true_list, k=k)
        recall_10 += recall_at_k(rec_list, true_list, k=k)

    hit_rate_10 = hit_rate_10 / len(result_df)
    precision_10 = precision_10 / len(result_df)
    recall_10 = recall_10 / len(result_df)

    map_10 = map_k(
        result_df[rec_col].tolist(),
        result_df[true_col].tolist(),
        k=k,
        u=len(result_df)
    )

    return hit_rate_10, precision_10, recall_10, map_10

In [25]:
# для LightFM нужна матрица user-item
# поэтому сначала сделаем индексы пользователей и item

from scipy.sparse import csr_matrix

In [26]:
# делаем словари:
# user_id -> индекс
# item_id -> индекс

train_users = df_train_lfm['user_id'].unique()
train_items = df_train_lfm['item_id'].unique()

user2id = {}
item2id = {}

for i in range(len(train_users)):
    user2id[train_users[i]] = i

for i in range(len(train_items)):
    item2id[train_items[i]] = i

id2item = {}
for item_id, idx in item2id.items():
    id2item[idx] = item_id

print('users in matrix:', len(user2id))
print('items in matrix:', len(item2id))

users in matrix: 5054
items in matrix: 4642


In [27]:
# теперь строим матрицу взаимодействий для train
# для простоты делаем бинарную матрицу:
# если взаимодействие было, ставим 1

train_pairs = df_train_lfm[['user_id', 'item_id']].drop_duplicates().copy()

rows = train_pairs['user_id'].map(user2id).values
cols = train_pairs['item_id'].map(item2id).values
vals = np.ones(len(train_pairs))

train_matrix = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user2id), len(item2id))
)

print(train_matrix.shape)
print(train_matrix.nnz)

(5054, 4642)
35107


In [28]:
# сохраним историю пользователя
# это нужно, чтобы не рекомендовать то, что он уже видел

user_history = {}

for user_id, group in train_pairs.groupby('user_id'):
    user_history[user_id] = set(group['item_id'].tolist())

print('users with history:', len(user_history))

users with history: 5054


In [29]:
# еще сохраним список популярных item
# он пригодится, если у модели не хватит нормальных рекомендаций

global_top = df_train_lfm['item_id'].value_counts().index.tolist()

print('top 10 popular items:')
print(global_top[:10])

top 10 popular items:
[9728, 13865, 3734, 10440, 15297, 142, 7571, 8636, 3182, 4151]


Обучение LightFM

In [30]:
#  начинаем обучение LightFM

model_lfm = LightFM(
    no_components=20,
    # loss='warp' хорошо подходит для top-k рекомендаций
    loss='warp',
    learning_rate=0.05,
    random_state=42
)

model_lfm.fit(
    train_matrix,
    epochs=10,
    num_threads=1
)

In [31]:
# функция рекомендаций для одного пользователя
# здесь берем все item из train-матрицы
# считаем для них score через model.predict
# и убираем то, что пользователь уже видел

def recommend_for_user_lightfm(user_id, n=10):
    if user_id not in user2id:
        return global_top[:n]

    user_idx = user2id[user_id]
    item_idx_list = np.arange(len(item2id))

    scores = model_lfm.predict(
        user_ids=user_idx,
        item_ids=item_idx_list,
        num_threads=1
    )

    seen_items = user_history.get(user_id, set())

    pairs = []

    for item_idx in range(len(scores)):
        item_id = id2item[item_idx]
        score = scores[item_idx]

        if item_id not in seen_items:
            pairs.append((item_id, score))

    pairs = sorted(pairs, key=lambda x: x[1], reverse=True)

    recs = []

    for item_id, score in pairs:
        recs.append(item_id)

        if len(recs) == n:
            break

    # если вдруг не хватило рекомендаций, добиваем популярными item
    if len(recs) < n:
        for item_id in global_top:
            if item_id not in recs and item_id not in seen_items:
                recs.append(item_id)

            if len(recs) == n:
                break

    return recs

In [32]:
# маленькая проверка на одном пользователе

test_user = data_test_group['user_id'].iloc[0]

print(test_user)
print(recommend_for_user_lightfm(test_user, n=10))

259
[13865, 9728, 142, 5434, 15297, 10440, 3734, 1844, 14431, 4495]


In [33]:
# теперь делаем рекомендации для всех пользователей из test

result_lfm = data_test_group[['user_id']].copy()

lfm_recs = []

for user_id in result_lfm['user_id']:
    recs = recommend_for_user_lightfm(user_id, n=10)
    lfm_recs.append(recs)

result_lfm['lightfm_recommendation'] = lfm_recs
result_lfm = result_lfm.merge(data_test_group, on='user_id', how='left')

result_lfm.head()

,user_id,lightfm_recommendation,item_id
0,259,"[13865, 9728, 142, 5434, 15297, 10440, 3734, 1...","[10509, 10772]"
1,655,"[6809, 7102, 9728, 4436, 3734, 142, 10440, 138...","[11188, 5199]"
2,753,"[13865, 9728, 3734, 7829, 10440, 11237, 15297,...","[9327, 8350]"
3,835,"[3734, 7310, 13865, 7582, 2956, 7571, 2954, 31...","[5434, 11640, 10878]"
4,960,"[3734, 9728, 15297, 4151, 4436, 13865, 1844, 4...","[8636, 12770, 6809]"


In [34]:
# считаем метрики для LightFM


lfm_hit_rate_10, lfm_precision_10, lfm_recall_10, lfm_map_10 = calc_metrics(
    result_lfm,
    rec_col='lightfm_recommendation',
    true_col='item_id',
    k=10
)

print('lightfm hit_rate@10 =', lfm_hit_rate_10)
print('lightfm precision@10 =', lfm_precision_10)
print('lightfm recall@10 =', lfm_recall_10)
print('lightfm map@10 =', lfm_map_10)

lightfm hit_rate@10 = 0.17590027700831026
lightfm precision@10 = 0.0195686584883259
lightfm recall@10 = 0.07214181577557097
lightfm map@10 = 0.07022780949868386


In [35]:
# соберем результаты в одну таблицу

result_table = pd.DataFrame({
    'model': ['LightFM'],
    'hit_rate@10': [lfm_hit_rate_10],
    'precision@10': [lfm_precision_10],
    'recall@10': [lfm_recall_10],
    'map@10': [lfm_map_10]
})

result_table

,model,hit_rate@10,precision@10,recall@10,map@10
0,LightFM,0.1759,0.019569,0.072142,0.070228



Вывод:
1. LightFM обучился на матрице взаимодействий user-item.
2. Для каждого пользователя мы получили top-10 рекомендаций.
3. Качество модели оценили по hit_rate@10, precision@10, recall@10 и map@10.
4. LightFM по метрикам показала себя лучше UserKNN, ItemKNN и ALS.